# Stage 16 — Fortran force-density normalization audit
**Objective:** Trace the active acceleration update in supplied `Console2.f90` and reproduce its force-density convention without assuming this source is Tu Bowen's exact Chapter 3 executable. This notebook is an auditable unit-level source translation **not** a complete validated impact simulation.

**Key finding:** The Fortran `pforce33` contribution computed as `bc * stretch * vol` is summed into `pforce` and divided by material density (`dens`, `dengs`). In contrast, previous Python prototype used `F/m_node`. This is a numerical scale difference when `F` is actually the Fortran force-density contribution.

**Important:** `Console2.f90` sets `dx=0.019 m`, `dt=3e-6 s`, `dens=2500 kg/m³`, `dengs=7850 kg/m³`, and its loading geometry differs from Bowen's thesis. This audit tests that Fortran, not claims of exact Bowen results.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from math import pi, sqrt
print('NumPy', np.__version__)

## 1. Source statements (transcribed with line references)
From `Console2.f90`:
- Lines 92–100: `dx=19.0d-3`, `delta=3.015d0*dx`, `emod=3.05d10`, `vol=dx**3`, `bc=12*emod/(pi*delta**4)`.
- Around lines 780–818: `dforce=bc*sssss*vol*direction`, add to `pforce33(i,:)`, subtract its 8-node interpolated fractions from concrete neighbors.
- Lines 832–834: `pforce=pforce11+pforce22+pforce33`.
- Lines 895–920: `acc=(pforce+bforce-cn*vel)/dens` (concrete) or `/dengs` (steel); then `vel += acc*dt`, `disp += vel*dt`.
- Around line 200: `vol1=dx*(530.66e-6)` is used in *steel rod element* `pforce22` calculation, not in the printed `pforce33` coupling expression.

The `pforce` arrays therefore act as force **per reference volume** for the time update. A dimensional interpretation of `bc` as N/m⁶ makes `bc*vol` N/m³. The equal/opposite `pforce33` entries cannot be summed as **physical forces** without multiplying by associated volumes and carefully distinguishing the steel and concrete control volumes.

In [ ]:
# Literal initialization from the available Console2.f90 (NOT Bowen Chapter 3)
dx = 19e-3
E = 3.05e10
delta = 3.015*dx
V = dx**3
bc = 12*E/(pi*delta**4)
rho_c, rho_s = 2500.0, 7850.0
cn = 3e6
dt = 3e-6
pd.DataFrame({'symbol':['dx','delta','vol','bc','dens','dengs','cn','dt'],
'value':[dx,delta,V,bc,rho_c,rho_s,cn,dt],
'source':['Fortran']*8})

## 2. Force density and acceleration: exact active source convention
For an elastic interaction the Appendix C code evaluates `q = bc*s*vol*n`, where `n` is the deformed bond unit vector. The literal integrator does **not** divide this value by an effective nodal mass; it divides assembled `pforce` by density.

For purposes of a paired numerical experiment, assume a single source coupling interaction and zero velocity/damping. This is not a physical element-level calibration.

In [ ]:
def source_c_elastic_force_density(bc, stretch, volume, direction):
    direction=np.asarray(direction,float)
    assert np.isclose(np.linalg.norm(direction),1)
    return bc*stretch*volume*direction

def fortran_acceleration(pforce, rho, bforce=None, velocity=None, cn=0.):
    q=np.asarray(pforce,float)
    b=np.zeros_like(q) if bforce is None else np.asarray(bforce,float)
    v=np.zeros_like(q) if velocity is None else np.asarray(velocity,float)
    return (q+b-cn*v)/rho

s=1e-6
q=source_c_elastic_force_density(bc,s,V,[1,0,0])
a_fortran_c=fortran_acceleration(q,rho_c)
a_fortran_s=fortran_acceleration(q,rho_s)
# Incorrect if q has the unit/meaning of source pforce33:
m_s=rho_s*V
a_previous=q/m_s
print(f'bc={bc:.6e}, vol={V:.6e}, pforce33={q[0]:.6e} N/m³')
print(f'Fortran steel acceleration: {a_fortran_s[0]:.6e} m/s²')
print(f'Incorrect lumped-mass acceleration: {a_previous[0]:.6e} (unit-inconsistent)')
print(f'Ratio previous/source: {a_previous[0]/a_fortran_s[0]:.6e} = 1/vol')
assert np.isclose(a_previous[0]/a_fortran_s[0],1/V)
assert np.isfinite(a_fortran_s).all()

## 3. Transfer to multiple concrete interpolation nodes
Appendix C distributes the opposite `dforce` contribution using 8 interpolation weights. It conserves the **sum of force-density entries** (if the eight weights sum to one). That does **not automatically imply conservation of actual net force** when steel and concrete control volumes differ. Physical force checks must weight density-array entries by the appropriate material control volumes.

Thus, we should preserve the author's indexing and assembly for the **literal source** case, while recording separately whether the resulting algorithm conserves physical momentum.

In [ ]:
weights=np.array([.125]*8)
qsteel=q.copy(); qconcrete=-weights[:,None]*q[None,:]
residual_density=qsteel+qconcrete.sum(axis=0)
assert np.linalg.norm(residual_density)<1e-10
# Physical force resultant if steel rod and concrete point control volumes differ:
Vsteel=dx*(530.66e-6)  # Fortran's vol1 (rod element control volume)
physical_sum=qsteel*Vsteel+qconcrete.sum(axis=0)*V
print('sum force-density contributions:',residual_density)
print('volume-weighted physical net force:',physical_sum,'N')
print('NOTE: nonzero volume-weighted result means literal array-level antisymmetry is not by itself physical momentum conservation.')

## 4. A separately labeled physically consistent alternative
If physical nodal forces are desired, one must define physical interaction force `F_int` and apply `+F_int/m_steel`, `-N_a*F_int/m_concrete_a`; or use force densities `F_int/V_steel` and `-N_a*F_int/V_concrete_a` before dividing by material density. Do **not** use the raw Fortran `pforce33` vector as physical nodal force without establishing its intended units. The following test simply verifies algebraic equivalence of these two **physical** representations.

In [ ]:
F=np.array([12.,-3.,7.])
msteel=rho_s*Vsteel
mconc=rho_c*V
acc_nodal_s=F/msteel
acc_density_s=(F/Vsteel)/rho_s
acc_nodal_c=-weights[:,None]*F/mconc
acc_density_c=(-weights[:,None]*F/V)/rho_c
assert np.allclose(acc_nodal_s,acc_density_s)
assert np.allclose(acc_nodal_c,acc_density_c)
assert np.allclose(msteel*acc_nodal_s+(mconc*acc_nodal_c).sum(axis=0),0,atol=1e-11)
print('PASS: consistent nodal-force / force-density forms give identical accelerations and conserve resultant physical force.')

## 5. Safe solver migration checklist
1. **Keep raw `pforce33` separate from physical nodal forces.** Name it `coupling_force_density` until scaling is established.
2. **Source-literal mode:** use active Fortran update `acc = (pforce+bforce-cn*vel)/rho`, matching the original program's force-density convention. Match all `pforce11`, `pforce22`, `pforce33` units before combining.
3. **Physically consistent mode:** convert each steel–concrete interaction once into a *physical* force with a justified interface measure; distribute `+F` and `-N_a F`, and divide by actual point masses. This is a **model correction**, not necessarily Bowen-equivalent.
4. Preserve `vol1` for steel rod force-density construction separately from `vol` for coupling; don't substitute them blindly.
5. Inspect physical momentum, energy, and 1-degree-of-freedom tangent stiffness/time step; source-equivalent code can still contain numerical defects.
6. Do not change the Chapter 3 parameter set or claim that a short-run test reproduces `359.8 kN` and `22.70 mm`.

**Open uncertainties:** full Bowen Chapter 3 source, exact PDROD quadrature/normalization, `point` mapping, geometry, contact, and which source program (if any) generated his thesis figures.

In [ ]:
# Deterministic regression summary
checks={
'Fortran density acceleration is finite': bool(np.isfinite(a_fortran_s).all()),
'Old mass scaling differs by exactly inverse volume': bool(np.isclose(a_previous[0]/a_fortran_s[0],1/V)),
'Appendix C interpolatory force-density balance': bool(np.linalg.norm(residual_density)<1e-10),
'Physical nodal-force and physical force-density accelerations agree': bool(np.allclose(acc_nodal_s,acc_density_s) and np.allclose(acc_nodal_c,acc_density_c)),
'Physical reformulation conserves resultant force': bool(np.allclose(msteel*acc_nodal_s+(mconc*acc_nodal_c).sum(axis=0),0,atol=1e-11))}
for name, ok in checks.items(): print(('PASS' if ok else 'FAIL'), name)
assert all(checks.values())
OUT=Path('bowen_stage16_outputs');OUT.mkdir(exist_ok=True)
pd.DataFrame([{'check':k,'passed':v} for k,v in checks.items()]).to_csv(OUT/'regression_checks.csv',index=False)
print('Saved',OUT/'regression_checks.csv')